
#Bronze Ingestion Framework

##Objective -
- load source files from azure container
- enforce schema contarcts
- add bronze audit metadata columns
- write to bronze delta tables
- register tables in unity catalog

In [0]:
# -------------------------------------
# environment configuration
# -------------------------------------

catalog_name = "cdl"
bronze_schema_name = "bronze"

storage_account_name = "datalakedevesh"

source_container_name = "source_cdl"
destination_container_name = "destination_cdl"

# unity catalog source volume paths

landing_volume_path = f"/Volumes/{catalog_name}/{bronze_schema_name}/landing_files/"

# direct cloud paths
source_base_path = f"abfss://{source_container_name}@{storage_account_name}.dfs.core.windows.net/"
bronze_base_path = f"abfss://{destination_container_name}@{storage_account_name}.dfs.core.windows.net/bronze/"

print("Catalog:",catalog_name)
print("Bronze Schema:",bronze_schema_name)
print("Landing volume path:",landing_volume_path)
print("Source cloud path:",source_base_path)
print("Destination cloud path:",bronze_base_path)



In [0]:
spark.sql(f"show schemas in {catalog_name}").display()
spark.sql(f"show volumes in {catalog_name}.{bronze_schema_name}").display()

In [0]:
display(dbutils.fs.ls(landing_volume_path))

In [0]:
source_files = [file for file in (dbutils.fs.ls(landing_volume_path)) if file.name.endswith(".json")]

display(source_files)

In [0]:
# file pareser - call_Activity_2026_06_15 to call_activity - 2026_06_15

import re

def parse_source_file_name(file_name):
    pattern = r"^(.*)_(\d{4}_\d{2}_\d{2})\.json$"
    match = re.match(pattern, file_name)
    if not match:
        raise ValueError(f"Invalid file name pattern: {file_name}")
    entity_name = match.group(1)
    date = match.group(2)
    return entity_name, date

# a,b = parse_source_file_name("call_activity-2026_06_15.json")
# print(f"entity is {a} and date is {b}")

In [0]:
# metadata creation for ingestion fw

discovered_files = []

for file in source_files:
    entity_name, date = parse_source_file_name(file.name)
    discovered_files.append({
        "source_file_path": file.path,
        "source_file_name": file.name,
        "source_entity": entity_name,
        "source_file_date": date,
        "file_format": "json"
    })
display(discovered_files)

In [0]:
spark.read.text(discovered_files[0]["source_file_path"]).display()

In [0]:
[file["source_entity"] for file in discovered_files]

In [0]:
# print(discovered_files)

df_sample = spark.read.json(discovered_files[0]["source_file_path"])
df_sample.printSchema()

In [0]:
# schema contract disctionary

from pyspark.sql.types import *

schema_contracts = {
    "call_activity":[
        {"column_name":"call_id","data_type":"string","nullable":False},
        {"column_name":"call_date","data_type":"string","nullable":False},
        {"column_name":"call_status","data_type":"string","nullable":False},
        {"column_name":"call_type","data_type":"string","nullable":False},
        {"column_name":"duration_minutes","data_type":"long","nullable":False},
        {"column_name":"hcp_id","data_type":"string","nullable":False},
        {"column_name":"product_id","data_type":"string","nullable":False},
        {"column_name":"source_file_date","data_type":"string","nullable":False},
        {"column_name":"source_system","data_type":"string","nullable":False},
        {"column_name":"territory_id","data_type":"string","nullable":False}
    ]
}

print(schema_contracts)